In [ ]:
#  importing libraries
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import OllamaLLM # connects Python to the Ollama app
from langchain_core.prompts import PromptTemplate

#  loading resources
# Loading the embedding model and the saved FAISS index 
print("Loading FAISS index...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.load_local("faiss_index_react", embedding_model, allow_dangerous_deserialization=True)

# Initialising llama 3.2 model running in the background Ollama app
# temperature=0 is for the llm to be strict and factual, and not be creative
print("Connecting to Llama 3.2...")
llm = OllamaLLM(model="llama3.2", temperature=0)

#  defining the grading prompt
# telling ai how to think
prompt_template = """
You are an strict academic assistant. Your task is to check if a specific Guideline is covered in the Course Content.

Guideline to check: "{guideline}"

Course Content Context (Retrieved from documents): 
"{context}"

INSTRUCTIONS:
1. Analyze if the context actually supports the guideline.
2. If it is covered, explain HOW.
3. If it is NOT covered, say "Not Covered".
4. Give a short explanation.

ANSWER:
"""

prompt = PromptTemplate(
    input_variables=["guideline", "context"],
    template=prompt_template
)

#  the full pipeline function
def check_guideline(guideline_text):
    print(f"\n--- Analyzing Guideline: {guideline_text} ---")
    
    #  Retrieving relevant content - RAG 
    # getting the top 3 chunks that might contain the answer
    results = vector_db.similarity_search(guideline_text, k=3)
    
    # Combining the text of the 3 chunks into one big string
    context_text = "\n\n".join([doc.page_content for doc in results])
    
    #  Generating the answer using the LLM   
    final_prompt = prompt.format(guideline=guideline_text, context=context_text)
    
    # Getting the response
    response = llm.invoke(final_prompt)
    
    return response

# testing hardcoded guideline
test_guideline = "The course must cover the syntax of conditional statements like if-else."

result = check_guideline(test_guideline)
print("\n--- ANALYSIS RESULT ---")
print(result)

Loading FAISS index...
Connecting to Llama 3.2...

--- Analyzing Guideline: The course must cover the syntax of conditional statements like if-else. ---

--- ANALYSIS RESULT ---
The Guideline "The course must cover the syntax of conditional statements like if-else" is partially covered in the Course Content Context.

The Course Content Context explicitly mentions the following types of conditional statements:

1. If statement: The context states that the if statement executes some code if one condition is true.
2. If-else statement: The context explains that the if-else statement executes some code if a condition is true and another code if that condition is false.
3. If...elseif....else statement: The context mentions that this type of statement executes different codes for more than two conditions.

However, it does not explicitly cover the syntax of the switch statement, which is also a type of conditional statement. While the context provides an example of how to use the switch sta